In [1]:
import numpy as np

In [2]:
rng = np.random.default_rng(42) #seed

In [5]:
shots = rng.random(10) < 0.8
shots

array([ True, False,  True, False,  True,  True,  True,  True, False,
        True])

In [6]:
shots.mean()

np.float64(0.7)

In [7]:
for n in [10, 100, 1000, 100000]:
    print(n, (rng.random(n) < 0.8).mean())

10 0.8
100 0.89
1000 0.794
100000 0.80041


Small samples are noisy and large samples settle down. That one fact (the Law of Large Numbers) is why simulation works: run something enough times and the fraction converges to the probability.



In [9]:
n = 1_000_000
d1 = rng.integers(1, 7, n)            # die 1
d2 = rng.integers(1, 7, n)            # die 2

A = d1 == 6
B = d2 == 6
print("P(A and B) sim:", (A & B).mean(), " formula:", 1/6 * 1/6)
print("P(A or B)  sim:", (A | B).mean(), " formula:", 1/6 + 1/6 - 1/36)
print("P(no six in 2) sim:", (~A & ~B).mean(), " formula:", (5/6)**2)

P(A and B) sim: 0.027774  formula: 0.027777777777777776
P(A or B)  sim: 0.304983  formula: 0.3055555555555555
P(no six in 2) sim: 0.695017  formula: 0.6944444444444445


Bernoulli(p): one yes/no trial

In [10]:
x = rng.random(10_000) < 0.75     # 10,000 free throws by a 75% shooter
x.mean()                          # ≈ 0.75

np.float64(0.7495)

Binomial(n, p): number of successes in n independent trials


In [11]:
n, p = 10, 0.75

# simulation: 100,000 nights of 10 free throws
makes = rng.binomial(n, p, 100_000)
sim_pmf = np.bincount(makes, minlength=n + 1) / len(makes)

from math import comb
formula = np.array([comb(n, k) * p**k * (1 - p)**(n - k) for k in range(n + 1)])

for k in range(n + 1):
    print(k, round(sim_pmf[k], 4), round(formula[k], 4))   # they match

0 0.0 0.0
1 0.0 0.0
2 0.0004 0.0004
3 0.0029 0.0031
4 0.0162 0.0162
5 0.0578 0.0584
6 0.1464 0.146
7 0.2501 0.2503
8 0.2839 0.2816
9 0.1866 0.1877
10 0.0556 0.0563


Poisson(λ): count of events in a fixed window


In [12]:
lam = 1.5                                   # a team averaging 1.5 goals
binom_version = rng.binomial(9000, lam / 9000, 200_000)
poisson_version = rng.poisson(lam, 200_000)
print(np.bincount(binom_version)[:7] / 200_000)
print(np.bincount(poisson_version)[:7] / 200_000)   # nearly identical

[0.222145 0.33442  0.251875 0.125425 0.04737  0.01444  0.003445]
[0.22468  0.334545 0.249555 0.12561  0.046885 0.01407  0.00377 ]


Geometric(p): trials until the first success


In [13]:
waits = rng.geometric(0.15, 100_000)
waits.mean()     # ≈ 6.67 = 1/0.15

np.float64(6.66474)

Continuous distributions

In [15]:
x = rng.normal(loc=200, scale=8, size=1_000_000)    # NBA heights in cm, roughly
for k in [1, 2, 3]:
    print(k, "sigma:", (np.abs(x - 200) < k * 8).mean())

1 sigma: 0.682752
2 sigma: 0.954451
3 sigma: 0.997336


Exponential(λ): waiting time between Poisson events


In [16]:
rate = 2.8 / 90                       # ~2.8 goals per 90 minutes
gaps = rng.exponential(1 / rate, 100_000)
gaps.mean()                           # ≈ 32 minutes between goals
(gaps > 90).mean()                    # chance of a 90-minute goal drought ≈ e^{-2.8} ≈ 0.06

np.float64(0.06099)

scipy.stats: the toolbox

In [17]:
from scipy import stats
X = stats.poisson(1.5)          # a "frozen" distribution
X.pmf(2)                        # P(X = 2)            (pdf for continuous)
X.cdf(2)                        # P(X <= 2)
X.sf(2)                         # P(X > 2) = 1 - cdf  ("survival function")
X.ppf(0.9)                      # the value with 90% below it (inverse CDF / quantile)
X.mean(), X.var()
X.rvs(size=10, random_state=0)  # random samples

N = stats.norm(loc=200, scale=8)
N.cdf(210) - N.cdf(190)         # P(190 <= height <= 210) ≈ 0.79
stats.binom(10, 0.75).pmf(10)   # P(all 10 free throws) ≈ 0.056

np.float64(0.056313514709472656)

Monte Carlo: answer anything by simulating it


In [18]:
n = 1_000_000
h = rng.poisson(1.6, n)
a = rng.poisson(1.1, n)
print("home win:", (h > a).mean())   # ≈ 0.49
print("draw:    ", (h == a).mean())  # ≈ 0.25
print("away win:", (h < a).mean())   # ≈ 0.26
print("over 2.5 goals:", (h + a > 2.5).mean())
print("both teams score:", ((h > 0) & (a > 0)).mean())

home win: 0.49043
draw:     0.24874
away win: 0.26083
over 2.5 goals: 0.506078
both teams score: 0.532085
